# Analisis Sentimen menggunakan Classical NLP
## TF-IDF dan Perbandingan Naive Bayes, Logistic Regression, dan SVM

## Identitas
Nama: Mukaromah

NIM: 240535602618

Offering: Teknik Informatika C

## Import Library

Pada tahap ini saya menggunakan beberapa library, yaitu **pandas** dan **numpy** untuk pengolahan data, **re** untuk membersihkan teks dengan regex, **Sastrawi** untuk stopword removal dan stemming bahasa Indonesia, serta **scikit-learn** untuk TF-IDF, train/test split, ketiga model klasifikasi, dan evaluasi.

In [ ]:
!pip install --quiet Sastrawi scikit-learn

import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 10.9 MB/s eta 0:00:00


## 1. Dataset

Dataset yang saya pakai adalah **Data Komentar Tiktok Progres IKN**, diambil dari Kaggle: https://www.kaggle.com/datasets/najmarafifah/analisis-sentimen-komentar-tiktok-progres-ikn

Dataset ini berisi komentar TikTok berbahasa Indonesia tentang progres IKN yang sudah dilabeli langsung sebagai positif atau negatif oleh pembuat dataset, jadi saya tidak perlu melakukan konversi dari rating maupun membuat label sendiri.

### Load Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)
df.head()

### Pemeriksaan Dataset

Pada tahap ini saya memeriksa ukuran, struktur kolom, tipe data, dan missing value dari dataset sebelum diproses lebih lanjut.

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

In [ ]:
df.info()

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nJumlah baris duplikat:", df.duplicated().sum())

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print("Shape setelah menghapus duplikat:", df.shape)

Dataset berisi 1.471 baris setelah 1 baris duplikat dihapus, terdiri dari kolom `text` (komentar TikTok) dan `label` (sentimen positif/negatif). Tidak ada missing value. Jumlah baris ini sudah memenuhi syarat minimal sekitar 1.000 baris.

### Distribusi Label

In [ ]:
distribusi_label = df["label"].value_counts()
persentase_label = df["label"].value_counts(normalize=True) * 100

print(distribusi_label)
print()
print(persentase_label.round(2))

Distribusi label adalah negatif sekitar 52,96% dan positif sekitar 47,04%. Rasio ini jauh dari ambang 80:20, sehingga dataset tidak timpang dan `class_weight='balanced'` tidak diperlukan untuk model Logistic Regression. Bahasa pada seluruh komentar juga konsisten, yaitu bahasa Indonesia informal.

## 2. Text Preprocessing

Preprocessing dilakukan melalui lima tahap berurutan, yaitu cleaning, case folding, tokenisasi, stopword removal, dan stemming. Kata negasi penting seperti "tidak", "bukan", "jangan", dan "nggak" sengaja saya pertahankan dan tidak ikut dihapus pada tahap stopword removal, karena kata ini bisa membalik makna sentimen sebuah kalimat.

### Cleaning

Pada tahap ini saya menghapus URL, mention (`@username`), angka, dan tanda baca menggunakan regex. Fungsi ini saya buat umum (mencakup URL dan mention) meskipun pada dataset ini kedua pola tersebut jarang ditemukan, karena komentar TikTok pada praktiknya tetap bisa mengandung pola seperti itu.

In [ ]:
def cleaning(text):
    text = re.sub(r"http\S+|www\.\S+", " ", text)   # URL
    text = re.sub(r"@\w+", " ", text)                # mention
    text = re.sub(r"\d+", " ", text)                 # angka
    text = re.sub(r"[^a-zA-Z\s]", " ", text)         # tanda baca & simbol
    text = re.sub(r"\s+", " ", text).strip()
    return text

contoh_teks = df["text"].iloc[63]
print("Sebelum :", contoh_teks)
print("Sesudah :", cleaning(contoh_teks))

### Case Folding

Tahap ini mengubah seluruh huruf menjadi huruf kecil, supaya variasi penulisan seperti "IKN" dan "ikn" dihitung sebagai token yang sama.

In [ ]:
def case_folding(text):
    return text.lower()

print("Sebelum :", cleaning(contoh_teks))
print("Sesudah :", case_folding(cleaning(contoh_teks)))

### Tokenisasi

Teks yang sudah bersih dipecah menjadi daftar token berdasarkan spasi.

In [ ]:
def tokenize(text):
    return text.split()

contoh_token = tokenize(case_folding(cleaning(contoh_teks)))
print(contoh_token)

### Stopword Removal

Saya menggunakan daftar stopword bahasa Indonesia dari **Sastrawi**, lalu mengeluarkan kata negasi ("tidak", "bukan", "jangan", "nggak", "gak", "ga", "tak", "kagak") dari daftar tersebut supaya kata-kata ini tetap dipertahankan saat penghapusan stopword.

In [ ]:
negasi = {"tidak", "bukan", "jangan", "nggak", "gak", "ga", "tak", "kagak"}

stopword_factory = StopWordRemoverFactory()
daftar_stopword = set(stopword_factory.get_stop_words())
daftar_stopword_final = daftar_stopword - negasi

print("Jumlah stopword awal:", len(daftar_stopword))
print("Jumlah stopword setelah kata negasi dikecualikan:", len(daftar_stopword_final))

In [ ]:
def remove_stopwords(tokens):
    return [t for t in tokens if t not in daftar_stopword_final]

contoh_tanpa_stopword = remove_stopwords(contoh_token)
print("Sebelum :", contoh_token)
print("Sesudah :", contoh_tanpa_stopword)

Untuk memastikan kata negasi tidak ikut terhapus, saya cek dengan contoh kalimat lain yang mengandung kata "jangan".

In [ ]:
contoh_negasi = df["text"].iloc[3]
token_negasi = tokenize(case_folding(cleaning(contoh_negasi)))

print("Sebelum :", token_negasi)
print("Sesudah :", remove_stopwords(token_negasi))

### Stemming / Lemmatization

Untuk bahasa Indonesia, saya menggunakan stemming dengan **Sastrawi**, karena library lemmatization standar seperti WordNet hanya mendukung bahasa Inggris. Sastrawi memotong imbuhan (awalan, akhiran, sisipan) sehingga kata seperti "menempati" menjadi "tempat".

In [ ]:
stemmer = StemmerFactory().create_stemmer()

def stemming(tokens):
    return [stemmer.stem(t) for t in tokens]

contoh_stemmed = stemming(contoh_tanpa_stopword)
print("Sebelum :", contoh_tanpa_stopword)
print("Sesudah :", contoh_stemmed)

### Hasil Preprocessing

Kelima tahap di atas saya gabungkan menjadi satu fungsi `preprocess()` dan diterapkan ke seluruh kolom `text`. Hasilnya disimpan pada kolom baru `text_bersih` yang akan dipakai sebagai input TF-IDF.

In [ ]:
def preprocess(text):
    text = cleaning(text)
    text = case_folding(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = stemming(tokens)
    return " ".join(tokens)

df["text_bersih"] = df["text"].apply(preprocess)

In [ ]:
df = df[df["text_bersih"].str.strip() != ""].reset_index(drop=True)
print("Shape setelah baris kosong hasil preprocessing dibuang:", df.shape)

In [ ]:
df[["text", "text_bersih"]].head(10)

Dari contoh di atas terlihat tanda baca, angka, dan stopword sudah hilang, sementara kata negasi seperti "jangan" tetap dipertahankan. Hasil kolom `text_bersih` ini yang akan dipakai sebagai input TF-IDF.

## 3. TF-IDF

TF-IDF dibentuk menggunakan `TfidfVectorizer` dengan `max_features=3000` dan `ngram_range=(1, 2)`, supaya kombinasi dua kata (seperti "tidak_bagus") juga ikut tertangkap sebagai fitur, tidak hanya kata tunggal.

Supaya tidak terjadi data leakage, train/test split saya lakukan terlebih dahulu sebelum TF-IDF dibentuk. `fit_transform()` hanya dipanggil pada data train, sedangkan data test hanya memakai `transform()` dengan vocabulary dan nilai IDF yang sudah dipelajari dari data train. Ukuran train dan test ditampilkan pada bagian 4.

In [ ]:
X = df["text_bersih"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Ukuran matriks TF-IDF train:", X_train_tfidf.shape)
print("Ukuran matriks TF-IDF test :", X_test_tfidf.shape)

## 4. Train/Test Split

Split dilakukan dengan proporsi 80% data train dan 20% data test, memakai `stratify=y` supaya proporsi label pada train dan test tetap seimbang, serta `random_state=42` yang sama dipakai untuk seluruh model agar hasil dapat dibandingkan secara adil.

In [ ]:
print("Ukuran data train:", X_train.shape[0])
print("Ukuran data test :", X_test.shape[0])
print()
print("Distribusi label train:")
print(y_train.value_counts())
print()
print("Distribusi label test:")
print(y_test.value_counts())

## 5. Model Training

Ketiga model dilatih menggunakan fitur TF-IDF yang sama, yaitu `X_train_tfidf`.

### Naive Bayes

Saya menggunakan `MultinomialNB` dengan parameter default, karena model ini memang didesain untuk fitur berbasis frekuensi atau bobot kata seperti TF-IDF.

In [ ]:
model_nb = MultinomialNB()
model_nb.fit(X_train_tfidf, y_train)

### Logistic Regression

Karena distribusi label (52,99% vs 47,01%) tidak terlalu timpang seperti yang sudah diperiksa pada bagian 1, saya tidak menggunakan `class_weight='balanced'` pada model ini.

In [ ]:
model_lr = LogisticRegression(max_iter=1000, random_state=42)
model_lr.fit(X_train_tfidf, y_train)

### SVM

Untuk `LinearSVC`, nilai `C` dipilih dengan menguji 0.1, 1, dan 10 memakai 5-fold cross-validation pada data training saja, supaya pemilihan parameter tidak mengintip data test.

In [ ]:
nilai_C = [0.1, 1, 10]
hasil_cv = {}

for c in nilai_C:
    model_cv = LinearSVC(C=c, random_state=42)
    skor = cross_val_score(model_cv, X_train_tfidf, y_train, cv=5, scoring="accuracy")
    hasil_cv[c] = skor.mean()
    print(f"C={c} -> rata-rata akurasi cross-validation: {skor.mean():.4f}")

C_terbaik = max(hasil_cv, key=hasil_cv.get)
print("\nNilai C terbaik:", C_terbaik)

In [ ]:
model_svm = LinearSVC(C=C_terbaik, random_state=42)
model_svm.fit(X_train_tfidf, y_train)

Nilai `C` terbaik berdasarkan cross-validation pada data training adalah **C=1**, sehingga model SVM final dilatih memakai nilai tersebut.

## 6. Prediction

Ketiga model yang sudah dilatih dipakai untuk memprediksi `X_test_tfidf`, dan hasilnya disimpan untuk dibandingkan pada tahap evaluasi.

In [ ]:
pred_nb = model_nb.predict(X_test_tfidf)
pred_lr = model_lr.predict(X_test_tfidf)
pred_svm = model_svm.predict(X_test_tfidf)

hasil_prediksi = pd.DataFrame({
    "text": X_test.values,
    "label_aktual": y_test.values,
    "pred_nb": pred_nb,
    "pred_lr": pred_lr,
    "pred_svm": pred_svm,
})
hasil_prediksi.head(10)

## 7. Evaluation

Karena hanya ada dua label (positif dan negatif), precision, recall, dan f1-score dihitung dengan `pos_label="positif"`.

In [ ]:
def evaluasi_model(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, pos_label="positif"),
        "Recall": recall_score(y_true, y_pred, pos_label="positif"),
        "F1-score": f1_score(y_true, y_pred, pos_label="positif"),
    }

skor_nb = evaluasi_model(y_test, pred_nb)
skor_lr = evaluasi_model(y_test, pred_lr)
skor_svm = evaluasi_model(y_test, pred_svm)

### Tabel Perbandingan

In [ ]:
tabel_perbandingan = pd.DataFrame({
    "Naive Bayes": skor_nb,
    "Logistic Regression": skor_lr,
    "SVM": skor_svm,
}).T

tabel_perbandingan.round(4)

### Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, nama, pred in zip(
    axes,
    ["Naive Bayes", "Logistic Regression", "SVM"],
    [pred_nb, pred_lr, pred_svm],
):
    cm = confusion_matrix(y_test, pred, labels=["negatif", "positif"])
    disp = ConfusionMatrixDisplay(cm, display_labels=["negatif", "positif"])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(nama)

plt.tight_layout()
plt.show()

Berdasarkan tabel perbandingan, **Naive Bayes** memperoleh accuracy dan f1-score tertinggi di antara ketiga model pada dataset ini. **Logistic Regression** memiliki precision tertinggi tetapi recall paling rendah, artinya model ini lebih sering memprediksi negatif ketika ragu, sehingga cukup banyak komentar positif yang justru salah diprediksi sebagai negatif (terlihat pada confusion matrix, 39 komentar positif salah diprediksi negatif). **SVM** berada di antara keduanya, dengan performa yang mendekati Naive Bayes namun sedikit lebih rendah pada accuracy.

## 8. Error Analysis

Pada tahap ini saya mengambil contoh data test yang salah diprediksi, baik oleh satu model maupun oleh ketiga model sekaligus.

In [ ]:
salah_nb = hasil_prediksi[hasil_prediksi["label_aktual"] != hasil_prediksi["pred_nb"]]
salah_lr = hasil_prediksi[hasil_prediksi["label_aktual"] != hasil_prediksi["pred_lr"]]
salah_svm = hasil_prediksi[hasil_prediksi["label_aktual"] != hasil_prediksi["pred_svm"]]

print("Jumlah salah prediksi Naive Bayes       :", len(salah_nb))
print("Jumlah salah prediksi Logistic Regression:", len(salah_lr))
print("Jumlah salah prediksi SVM                :", len(salah_svm))

In [ ]:
salah_ketiganya = hasil_prediksi[
    (hasil_prediksi["label_aktual"] != hasil_prediksi["pred_nb"]) &
    (hasil_prediksi["label_aktual"] != hasil_prediksi["pred_lr"]) &
    (hasil_prediksi["label_aktual"] != hasil_prediksi["pred_svm"])
]

print("Jumlah data yang salah diprediksi oleh ketiga model:", len(salah_ketiganya))
salah_ketiganya[["text", "label_aktual", "pred_nb", "pred_lr", "pred_svm"]].head(8)

In [ ]:
salah_beda = hasil_prediksi[
    ~hasil_prediksi.index.isin(salah_ketiganya.index) &
    ((hasil_prediksi["label_aktual"] != hasil_prediksi["pred_nb"]) |
     (hasil_prediksi["label_aktual"] != hasil_prediksi["pred_lr"]) |
     (hasil_prediksi["label_aktual"] != hasil_prediksi["pred_svm"]))
]

print("Jumlah data yang hanya salah pada sebagian model:", len(salah_beda))
salah_beda[["text", "label_aktual", "pred_nb", "pred_lr", "pred_svm"]].head(8)

Dari contoh yang salah diprediksi ketiga model, sebagian besar berupa kalimat dengan **sentimen tersirat** yang tidak memakai kata sentimen eksplisit. Misalnya komentar "jdi kepingin tinggal disana suasanya kayak adem gitu" (aktual positif) salah diprediksi negatif oleh ketiga model, karena kata seperti "kepingin" dan "adem" tidak banyak muncul sebagai penanda sentimen kuat di corpus training. Contoh lain seperti "akhirnya seperti meikarta atau hambalang" (aktual negatif, diprediksi positif oleh ketiga model) menunjukkan pola **sarkasme dan referensi tersirat**, di mana makna negatif muncul dari perbandingan dengan proyek lain yang gagal, bukan dari kata bermuatan sentimen secara langsung. TF-IDF tidak bisa menangkap pengetahuan di luar teks seperti ini.

Pada data yang hanya salah di sebagian model, pola yang muncul lebih banyak berkaitan dengan **sentimen campuran** dalam satu kalimat, misalnya kalimat yang mengandung pujian sekaligus kritik, sehingga model dengan ambang keputusan yang berbeda (seperti Logistic Regression yang punya precision tinggi tapi recall rendah) lebih mudah salah dibanding Naive Bayes.

Berdasarkan hasil eksperimen pada dataset ini, **Naive Bayes** adalah model yang paling robust, karena memiliki accuracy dan f1-score tertinggi, serta jumlah kesalahan prediksi paling sedikit dibanding Logistic Regression dan SVM. Hal ini kemungkinan karena Naive Bayes cocok dengan karakteristik fitur TF-IDF yang sparse dan performanya tidak terlalu bergantung pada tuning parameter, berbeda dengan SVM yang sensitif terhadap nilai `C` atau Logistic Regression yang cenderung bias ke kelas mayoritas ketika model kurang yakin.

## Kesimpulan

**Karakteristik dataset.** Dataset berisi 1.471 komentar TikTok berbahasa Indonesia tentang progres IKN, dengan label sentimen asli (bukan hasil konversi rating), terdiri dari 52,99% negatif dan 47,01% positif. Distribusi ini tidak timpang dan bahasa pada seluruh data konsisten, sehingga dataset memenuhi syarat untuk digunakan.

**Hasil preprocessing.** Preprocessing melalui lima tahap (cleaning, case folding, tokenisasi, stopword removal, stemming) berhasil membersihkan teks dari tanda baca, angka, dan stopword, sambil tetap mempertahankan kata negasi penting seperti "tidak" dan "jangan" yang krusial untuk analisis sentimen.

**Perbandingan ketiga model.** Naive Bayes memperoleh accuracy dan f1-score tertinggi, Logistic Regression memiliki precision tertinggi namun recall terendah, sedangkan SVM (dengan C=1 hasil cross-validation) berada di antara keduanya.

**Pola kesalahan.** Kesalahan prediksi pada ketiga model banyak terjadi pada kalimat dengan sentimen tersirat, sarkasme, dan referensi ke konteks di luar teks, karena TF-IDF hanya menangkap kemunculan kata, bukan makna implisit. Sentimen campuran dalam satu kalimat juga menjadi sumber kesalahan yang lebih sering muncul pada Logistic Regression dan SVM dibanding Naive Bayes.

**Model paling robust.** Berdasarkan hasil eksperimen pada dataset ini, Naive Bayes adalah model yang paling robust, karena konsisten unggul pada accuracy dan f1-score, serta jumlah kesalahan prediksinya paling sedikit dibanding dua model lainnya.